# 3 · Adding a tool (and the dynamic registry)

Demo 1 showed how a request *exposes* tools, and demo 2 *executed* them. But a
fixed set of tools is limiting. An agent should be able to **gain a new
capability** — this is "dynamic tools": the tool surface is **not fixed at
compile time**. The `ToolRegistry` is just an in-memory map: you can register,
replace, and unregister tools **while the agent is alive**.

We keep using the **file, web and calculator** families. Here we bolt a brand-
new *file* tool (`file_head`) directly onto the running registry.

This demo is **fully local**.


In [3]:
:dep agent_loop = { path = "/home/christian/Sandbox/agent-loop" }
:dep serde_json = "1"

use agent_loop::tools::{ToolRegistry, Tool, ToolResult, s_required, builtin_tools};
use serde_json::{json, Value};

let mut registry = ToolRegistry::with_builtins();
println!("initial tools ({}): {}", registry.len(),
    registry.all().iter().map(|t| t.name.as_str()).collect::<Vec<_>>().join(", "));


initial tools (7): file_read, file_write, file_list, bash_run, web_fetch, web_search, calc


### How to add a tool

A tool is two things: a **description** the model can read (name + JSON-schema
parameters) and an **executor** (the closure that does the work). `Tool::new`
bundles them. Below we add `file_head`, which returns the first `n` lines of a
file — part of the file-ops family, added live.


In [4]:

// The tool did NOT exist a moment ago; we register it at runtime.
registry.register(Tool::new(
    "file_head",
    "Return the first n lines of a text file.",
    s_required(json!({
        "path": {"type": "string"},
        "n": {"type": "integer"}
    }), &["path"]),
    |args| {
        let path = args.get("path").and_then(Value::as_str).unwrap_or("");
        let n = args.get("n").and_then(Value::as_i64).unwrap_or(5) as usize;
        let text = std::fs::read_to_string(path)?;
        let head: Vec<&str> = text.lines().take(n).collect();
        Ok(ToolResult::ok(head.join("\n")))
    },
));

println!("after adding file_head ({}): {}", registry.len(),
    registry.all().iter().map(|t| t.name.as_str()).collect::<Vec<_>>().join(", "));
println!("call it now ->
{}", registry.get("file_head").unwrap()
    .run(&json!({"path": "Cargo.toml", "n": 3})).unwrap().output);


after adding file_head (8): file_read, file_write, file_list, bash_run, web_fetch, web_search, calc, file_head



thread '<unnamed>' (918061) panicked at src/lib.rs:124:50:
called `Result::unwrap()` on an `Err` value: No such file or directory (os error 2)

Stack backtrace:
   0: anyhow::error::<impl core::convert::From<E> for anyhow::Error>::from
   1: ctx::run_user_code_2::{{closure}}::{{closure}}
   2: agent_loop::tools::Tool::run
   3: run_user_code_2
   4: evcxr::runtime::Runtime::run_loop
   5: evcxr::runtime::runtime_hook
   6: evcxr_jupyter::main
   7: std::sys::backtrace::__rust_begin_short_backtrace
   8: std::rt::lang_start::{{closure}}
   9: std::rt::lang_start_internal
  10: main
  11: __libc_start_call_main
             at ./csu/../sysdeps/nptl/libc_start_call_main.h:58:16
  12: __libc_start_main_impl
             at ./csu/../csu/libc-start.c:360:3
  13: _start
stack backtrace:
   0: __rustc::rust_begin_unwind
             at /rustc/e408947bfd200af42db322daf0fadfe7e26d3bd1/library/std/src/panicking.rs:689:5
   1: core::panicking::panic_fmt
             at /rustc/e408947bfd200af42db3

### Replace a handler at runtime

Same name, new behaviour: drop-in upgrades without changing the call site. The
model never knows — it just continues emitting `file_head(...)` calls.


In [4]:

let before = registry.get("file_head").unwrap()
    .run(&json!({"path":"Cargo.toml","n":2})).unwrap().output;

// Same tool name, new handler: now it returns a file *snippet* instead.
registry.register(Tool::new(
    "file_head",
    "Return the first n lines of a text file, labelled.",
    s_required(json!({
        "path": {"type": "string"},
        "n": {"type": "integer"}
    }), &["path"]),
    |args| {
        let path = args.get("path").and_then(Value::as_str).unwrap_or("");
        let n = args.get("n").and_then(Value::as_i64).unwrap_or(5) as usize;
        let text = std::fs::read_to_string(path)?;
        let head: Vec<String> = text.lines().take(n).map(|l| format!("> {l}")).collect();
        Ok(ToolResult::ok(head.join("\n")))
    },
));

let after = registry.get("file_head").unwrap()
    .run(&json!({"path":"Cargo.toml","n":2})).unwrap().output;
println!("before replacement:
{before}\n");
println!("after  replacement:
{after}");


thread '<unnamed>' (855251) panicked at src/lib.rs:105:47:


called `Result::unwrap()` on an `Err` value: No such file or directory (os error 2)


Stack backtrace:


   0: anyhow::error::<impl core::convert::From<E> for anyhow::Error>::from


   1: ctx::run_user_code_2::{{closure}}::{{closure}}


   2: <unknown>


   3: <unknown>


   4: evcxr::runtime::Runtime::run_loop


### Unregister a tool

Removing a capability is just as easy. Once unregistered, calls to it report the
tool as unknown so the model can adapt.


In [5]:

registry.unregister("file_head");
println!("after removing file_head ({}): {}", registry.len(),
    registry.all().iter().map(|t| t.name.as_str()).collect::<Vec<_>>().join(", "));

let unknown = agent_loop::chat::ToolCall { id: "x".into(), name: "file_head".into(), arguments: "{}".into() };
let msg = agent_loop::executor::execute_one(&registry, &unknown);
println!("call to removed tool -> {}", msg.content.unwrap());


after removing file_head (7): file_read, file_write, file_list, bash_run, web_fetch, web_search, calc


call to removed tool -> [unknown tool] `file_head` is not registered


### Why this matters

The request in demo 1 serializes whatever is in the registry at request time
(`registry.as_chat_tools()`). So adding a tool dynamically *changes the next
request* the model sees — the model learns about the new tool on the very next
turn. The **calculator** `calc` tool shipped in the crate is itself just
something added to the registry this way. Dynamic + single/parallel execution
(demos 1–2) + the loop from demo 5 = a live, growing agent.
